In [0]:
%sql
CREATE OR REPLACE TABLE workspace.gold.dim_producto AS
SELECT producto_id, nombre, linea, cultivo_objetivo, vida_util_dias, sensible_humedad
FROM workspace.bronze.productos_raw;

CREATE OR REPLACE TABLE workspace.gold.dim_cliente AS
SELECT cliente_id, nombre, tipo, departamento AS departamento_cliente
FROM workspace.bronze.clientes_raw;

In [0]:
from pyspark.sql.functions import col, broadcast

ventas = spark.table("workspace.silver.ventas")
dim_prod = spark.table("workspace.gold.dim_producto")

fact = (
    ventas
    .join(broadcast(dim_prod.select("producto_id", "linea")), on="producto_id", how="left")
    .select(
        "pedido_id", "fecha_pedido", "cliente_id", "producto_id",
        "linea", "departamento_destino",
        "tm_programadas", "tm_fabricadas", "tm_vendidas",
        "precio_usd_tm", "es_precio_imputado",
        (col("tm_vendidas") * col("precio_usd_tm")).alias("valor_venta_usd"),
    )
)

fact.write.format("delta").mode("overwrite").saveAsTable("workspace.gold.fact_ventas")
print("gold.fact_ventas creada:", spark.table("workspace.gold.fact_ventas").count())

In [0]:
%sql
SELECT
  departamento_destino,
  ROUND(SUM(tm_vendidas), 1)     AS toneladas_vendidas,
  ROUND(SUM(valor_venta_usd), 0) AS valor_usd
FROM workspace.gold.fact_ventas
GROUP BY departamento_destino
ORDER BY toneladas_vendidas DESC;

In [0]:
%sql
SELECT month(fecha_pedido) AS mes,
       ROUND(SUM(tm_vendidas), 0) AS toneladas_vendidas
FROM workspace.gold.fact_ventas
GROUP BY month(fecha_pedido)
ORDER BY mes;

In [0]:
%sql
SELECT month(fecha_pedido) AS mes,
       ROUND(SUM(CASE WHEN year(fecha_pedido)=2023 THEN tm_vendidas END),0) AS ventas_2023,
       ROUND(SUM(CASE WHEN year(fecha_pedido)=2025 THEN tm_vendidas END),0) AS ventas_2025
FROM workspace.gold.fact_ventas
GROUP BY month(fecha_pedido)
ORDER BY mes;

In [0]:
%sql
SELECT
  ROUND(CORR(mensual.tm, mensual.lluvia), 3) AS correlacion_venta_lluvia
FROM (
  SELECT f.departamento_destino, year(f.fecha_pedido) AS anio, month(f.fecha_pedido) AS mes,
         SUM(f.tm_vendidas) AS tm,
         AVG(c.precip_real_mm) AS lluvia
  FROM workspace.gold.fact_ventas f
  JOIN workspace.bronze.clima_raw c
    ON f.departamento_destino = c.departamento
   AND year(f.fecha_pedido)  = c.anio
   AND month(f.fecha_pedido) = c.mes
  GROUP BY f.departamento_destino, year(f.fecha_pedido), month(f.fecha_pedido)
) AS mensual;

In [0]:
%sql
WITH bronze_dedup AS (
  SELECT pedido_id, departamento_destino, fecha_pedido, tm_vendidas
  FROM workspace.bronze.ventas_raw
  WHERE departamento_destino IS NOT NULL
  QUALIFY ROW_NUMBER() OVER (PARTITION BY pedido_id ORDER BY fecha_actualizacion DESC) = 1
)
SELECT
  CASE
    WHEN c.precip_real_mm < 100 THEN '1_baja (<100mm)'
    WHEN c.precip_real_mm < 170 THEN '2_media (100-170)'
    ELSE '3_alta (>170mm)'
  END AS banda_lluvia,
  COUNT(*)                       AS n,
  ROUND(AVG(c.precip_real_mm),0) AS lluvia_prom,
  ROUND(AVG(b.tm_vendidas),1)    AS tm_vendidas_prom
FROM bronze_dedup b
JOIN workspace.bronze.clima_raw c
  ON b.departamento_destino = c.departamento
 AND year(b.fecha_pedido)  = c.anio
 AND month(b.fecha_pedido) = c.mes
GROUP BY 1
ORDER BY 1;

In [0]:
%sql
DESCRIBE workspace.gold.fact_ventas;

In [0]:
from pyspark.sql import functions as F

fact = spark.table("workspace.gold.fact_ventas")

resumen = (
    fact
    .withColumn("mes", F.trunc("fecha_pedido", "MM")) 
    .groupBy("mes", "producto_id", "linea", "departamento_destino")
    .agg(
        F.countDistinct("pedido_id").alias("num_pedidos"),
        F.round(F.sum("tm_programadas"), 1).alias("tm_programadas"),
        F.round(F.sum("tm_fabricadas"), 1).alias("tm_fabricadas"),
        F.round(F.sum("tm_vendidas"),  1).alias("tm_vendidas"),
        F.round(F.sum("valor_venta_usd"), 0).alias("valor_usd"),
        F.round(100 * F.avg(F.col("es_precio_imputado").cast("int")), 1).alias("pct_precio_imputado"),
    )
    .withColumn("sobre_stock_tm", F.round(F.col("tm_fabricadas") - F.col("tm_vendidas"), 1))
    .withColumn(
        "precio_prom_usd_tm",
        F.when(F.col("tm_vendidas") > 0, F.round(F.col("valor_usd") / F.col("tm_vendidas"), 1))
    )
    .orderBy("mes", F.col("tm_vendidas").desc())
)

(resumen.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.gold.resumen_ventas_mensual"))

display(spark.table("workspace.gold.resumen_ventas_mensual"))